## Path Finding in the City of Austin

This notebook demonstrates pathfinding along the city of Austin street network using Xarray-spatial's `pathfinding` module.
The a_star_search function provides the shortest path between any two points.

#### Setup:

First, we'll need to import some packages: these include the basic array manipulation ones,  
as well as some geospatial-focused ones.
We'll also grab a few matplotlib functions for easy rendering.

In [ ]:
import geopandas
import numpy as np
import pandas as pd
import xarray as xa

from xrspatial import a_star_search


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, to_rgba
import numpy as np

def _shade(arr, cmap=None, alpha=None, min_alpha=None, span=None, how='linear'):
    arr = np.asarray(arr, dtype=np.float64)
    finite = np.isfinite(arr)
    if span is not None:
        lo, hi = span
    elif finite.any():
        lo = np.nanmin(arr); hi = np.nanmax(arr)
    else:
        lo = hi = 0.0
    norm = np.zeros_like(arr, dtype=np.float32)
    if hi > lo:
        norm = np.where(finite, (arr - lo) / (hi - lo), 0.0).astype(np.float32)
    if isinstance(cmap, (list, tuple)):
        cmap = LinearSegmentedColormap.from_list('c', list(cmap), N=256)
    elif cmap is None:
        cmap = plt.get_cmap('terrain')
    rgba = cmap(norm, bytes=True)
    a = np.where(finite, 255 if alpha is None else alpha, 0).astype(np.uint8)
    if min_alpha is not None:
        a = np.where(finite & (a < min_alpha), min_alpha, a).astype(np.uint8)
    rgba[..., 3] = a
    return rgba

def _stack(*layers):
    base = layers[0].astype(np.float64)
    for top in layers[1:]:
        a = (top[..., 3:4] / 255.0)
        rgb = top[..., :3] * a + base[..., :3] * (1 - a)
        out_a = np.maximum(top[..., 3], base[..., 3])
        base = np.concatenate([rgb, out_a[..., None]], axis=-1)
    return np.clip(base, 0, 255).astype(np.uint8)

def _show(rgba, bg=None, figsize=(10, 8)):
    if bg is not None:
        h, w = rgba.shape[:2]
        bgc = to_rgba(bg)
        bg_rgba = np.zeros((h, w, 4), dtype=np.uint8)
        bg_rgba[..., :3] = (np.array(bgc[:3]) * 255).astype(np.uint8)
        bg_rgba[..., 3] = 255
        rgba = _stack(bg_rgba, rgba)
    plt.figure(figsize=figsize)
    plt.imshow(rgba)
    plt.axis('off')
    plt.show()


### Load data

Now, we're ready to load up the data and transform it into a format we can work with.

To download the examples data, run the command `xrspatial examples` in your terminal. All the data will be stored in your current directory inside a folder named `xrspatial-examples`.

We'll start by opening the shapefile, transforming the crs (coordinate reference system) to the commonly-used longitude/latitude,  
and.

Now our data is ready to be aggregated to an xarray DataArray raster.

In [ ]:
streets = geopandas.read_file("../xrspatial-examples/data/geo_export_9c395dda-0b29-41ec-89b4-a51a898f7104.shp")
streets = streets.to_crs("EPSG:4326")
streets = streets.explode("geometry").reset_index(drop=True)


### Define study area (find range of x and y) and aggregate:

To finish off our set-up:
- We'll define a study area, with xmin, xmax, ymin, and ymax; this set the x, y coordinates we'll be using in our aggregate.
- We'll set up a matplotlib Canvas object, which provides an easy frame for setting up a new raster and aggregating data to it.
- Finally, we'll aggregate the streets data into a lines raster with Canvas.line.

- We also set up the start and goal point (y, x) coordinates, and set up a DataFrame and aggregation for visualization.

Some shading and stacking of all of this displays our complete setup below.

In [ ]:
xmin, ymin, xmax, ymax = (
    streets.geometry.bounds.minx.min(),
    streets.geometry.bounds.miny.min(),
    streets.geometry.bounds.maxx.max(),
    streets.geometry.bounds.maxy.max(),
)
xrange = (xmin, xmax)
yrange = (ymin, ymax)
xrange, yrange


In [ ]:
H, W = 600, 800
grid_xs = np.linspace(xrange[0], xrange[1], W)
grid_ys = np.linspace(yrange[1], xrange[0], H)
template = xa.DataArray(np.full((H, W), np.nan),
                        coords={'y': grid_ys, 'x': grid_xs}, dims=['y', 'x'])

# Rasterize the street lines onto the grid with the .xrs accessor
# (replaces ds.Canvas().line(streets_spd, geometry='geometry')).
from shapely.geometry import Point
street_agg = template.xrs.rasterize(streets, merge='last')
street_shaded = _shade(street_agg, cmap=['salmon', 'salmon'], min_alpha=255)

# Pick two locations: (lat, lon) pairs
start = (30.08214069, -97.73662282)
goal = (30.17656606, -97.63753489)

start_gdf = geopandas.GeoDataFrame(
    {'id': [1]}, geometry=[Point(start[1], start[0])], crs='EPSG:4326')
start_agg = template.xrs.rasterize(start_gdf, column='id', merge='last')
start_shaded = _shade(start_agg, cmap=['red', 'red'], min_alpha=255)

goal_gdf = geopandas.GeoDataFrame(
    {'id': [1]}, geometry=[Point(goal[1], goal[0])], crs='EPSG:4326')
goal_agg = template.xrs.rasterize(goal_gdf, column='id', merge='last')
goal_shaded = _shade(goal_agg, cmap=['lime', 'lime'], min_alpha=255)

_show(_stack(street_shaded, start_shaded, goal_shaded), bg='black')


### Shortest path using A* from start location to goal location

Now, we can do some pathfinding:

In `a_star_search`, we'll input the Austin city streets lines aggregate we built above, the start and goal point coordinates, and barriers:
    - Barriers defines all non-crossable points in the raster: for our streets raster, this includes all non-street areas, all of which have 0 set as their value. 

We've also set `snap-start` and `snap-goal` to `True`: this helps ensure the start and goal points are set correctly.

The result is a the shortest path al
    

In [ ]:
# find the path from start to goal,
# barriers are uncrossable cells. In this case, they are cells with a value of 0

path_agg = a_star_search(
    street_agg, start, goal, barriers=[0], snap_start=True, snap_goal=True
)

path_shaded = _shade(path_agg, cmap=['green', 'green'], min_alpha=255)
_show(_stack(street_shaded, path_shaded, start_shaded, goal_shaded), bg='black')
